# Deploy Hotel Booking Agent to Production with Amazon Bedrock AgentCore

In this notebook you will deploy a **production-ready hotel booking agent** to AWS using Amazon Bedrock AgentCore — step by step.

## What You Will Build

| Component | AWS Service | Purpose |
|-----------|------------|----------|
| Data layer | DynamoDB (3 tables) | Hotels, bookings, steering rules |
| Booking tools | Lambda (7 functions) | Search, book, pay, confirm, cancel, validate |
| Graph query tool | Lambda (1 function, VPC) | Cypher queries to Neo4j knowledge graph |
| Tool routing | AgentCore Gateway (MCP) | Semantic tool discovery |
| Agent | AgentCore Runtime | Hosts the Strands agent with Bedrock |

This brings together the techniques from all previous modules:
- **Graph-RAG** (Module 1) → `query_knowledge_graph` Lambda connecting to Neo4j
- **Semantic tool routing** (Module 2) → AgentCore Gateway with MCP semantic search
- **Neurosymbolic guardrails** (Module 4) → Steering rules stored in DynamoDB
- **Steering** (Module 5) → Agent self-corrects based on STEER messages

## Workshop vs Self-paced

This notebook adapts to your environment:
- **At an AWS event:** Neo4j runs on your Code Editor EC2. The notebook detects the private IP and Security Group from CloudFormation outputs automatically.
- **Self-paced:** Set `NEO4J_HOST` manually to your Neo4j instance (AuraDB URI, local Docker, etc). If no Neo4j is available, the notebook deploys without the graph query tool.

## Region: **us-east-1**

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("✅ Environment ready")

---
## Step 0: Configuration

Set up AWS clients and naming conventions. All resources use a `workshop-` or `hotel-booking-` prefix so you can identify and clean them up later.

In [ ]:
import boto3
import json
import time
import zipfile
import io
import os
import subprocess

# --- Configuration ---
REGION = os.environ.get("AWS_REGION", "us-east-1")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# Resource names (prefixed for easy cleanup)
HOTELS_TABLE = "workshop-Hotels"
BOOKINGS_TABLE = "workshop-Bookings"
STEERING_RULES_TABLE = "workshop-SteeringRules"
LAMBDA_ROLE_NAME = "workshop-LambdaExecutionRole"
AGENTCORE_ROLE_NAME = "workshop-AgentCoreExecutionRole"
GATEWAY_NAME = "HotelBookingGateway"
RUNTIME_NAME = "HotelBookingAgent"

# AWS clients
dynamodb = boto3.client("dynamodb", region_name=REGION)
dynamodb_resource = boto3.resource("dynamodb", region_name=REGION)
iam = boto3.client("iam")
lambda_client = boto3.client("lambda", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
agentcore_data = boto3.client("bedrock-agentcore", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)
ec2 = boto3.client("ec2", region_name=REGION)

# --- Detect Neo4j infrastructure ---
# At an AWS event: values are set in /etc/environment by the CloudFormation stack.
# We read them from environment variables first, then fall back to CloudFormation outputs.

NEO4J_HOST = os.environ.get("NEO4J_HOST") or os.environ.get("NEO4J_URI", "").replace("bolt://", "").split(":")[0] or None
NEO4J_SECRET_ARN = os.environ.get("NEO4J_SECRET_ARN", "")
SECURITY_GROUP_ID = os.environ.get("SECURITY_GROUP_ID", "")
SUBNET_IDS = []

# If not in env vars, try CloudFormation outputs (the workshop stack has Neo4jHost output)
if not NEO4J_HOST or not SECURITY_GROUP_ID:
    cfn = boto3.client("cloudformation", region_name=REGION)
    try:
        paginator = cfn.get_paginator("list_stacks")
        for page in paginator.paginate(StackStatusFilter=["CREATE_COMPLETE", "UPDATE_COMPLETE"]):
            for stack_summary in page["StackSummaries"]:
                stack_name = stack_summary["StackName"]
                try:
                    outputs = cfn.describe_stacks(StackName=stack_name)["Stacks"][0].get("Outputs", [])
                    out_map = {o["OutputKey"]: o["OutputValue"] for o in outputs}
                    if "Neo4jHost" in out_map:
                        NEO4J_HOST = NEO4J_HOST or out_map["Neo4jHost"]
                        NEO4J_SECRET_ARN = NEO4J_SECRET_ARN or out_map.get("Neo4jSecretArn", "")
                        SECURITY_GROUP_ID = SECURITY_GROUP_ID or out_map.get("SecurityGroupId", "")
                        break
                except Exception:
                    continue
            if NEO4J_HOST:
                break
    except Exception:
        pass

# Get default VPC subnets (needed for Neo4j Lambda VPC config)
try:
    vpcs = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])["Vpcs"]
    if vpcs:
        subs = ec2.describe_subnets(Filters=[{"Name": "vpc-id", "Values": [vpcs[0]["VpcId"]]}])["Subnets"]
        SUBNET_IDS = [s["SubnetId"] for s in subs[:2]]
except Exception:
    pass

# --- Self-paced override ---
# If Neo4j is not detected automatically, set these manually:
# NEO4J_HOST = "your-neo4j-host"         # e.g., "localhost" or "xxxxx.databases.neo4j.io"
# NEO4J_SECRET_ARN = ""                   # Create a secret with your Neo4j password
# SECURITY_GROUP_ID = "sg-xxxxx"          # Your VPC security group
# SUBNET_IDS = ["subnet-xxxxx"]           # Your VPC subnets

print(f"Account:          {ACCOUNT_ID}")
print(f"Region:           {REGION}")
print(f"Neo4j Host:       {NEO4J_HOST or 'Not detected (Neo4j Lambda will be skipped)'}")
print(f"Neo4j Secret ARN: {NEO4J_SECRET_ARN or 'N/A'}")
print(f"Security Group:   {SECURITY_GROUP_ID or 'N/A'}")
print(f"Subnets:          {SUBNET_IDS or 'N/A'}")

### Recovery: Re-run safe

Every cell in this notebook is **idempotent** — if a resource already exists, the cell will find it and continue. You can re-run any cell or restart the notebook from any point without breaking anything.

If you need to recover variables from a previous partial run, execute the cell below to load existing resource IDs:

In [ ]:
# --- Recovery: Load existing resources if they already exist ---
# Run this cell if you restarted the kernel and need to recover state

# IAM Roles
try:
    LAMBDA_ROLE_ARN = iam.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]
    print(f"Lambda role:    {LAMBDA_ROLE_ARN}")
except: 
    LAMBDA_ROLE_ARN = None
    print("Lambda role: not created yet")

try:
    AGENTCORE_ROLE_ARN = iam.get_role(RoleName=AGENTCORE_ROLE_NAME)["Role"]["Arn"]
    print(f"AgentCore role: {AGENTCORE_ROLE_ARN}")
except:
    AGENTCORE_ROLE_ARN = None
    print("AgentCore role: not created yet")

# Lambda functions
if 'LAMBDA_ARNS' not in dir() or not LAMBDA_ARNS:
    LAMBDA_ARNS = {}
for tool_name in ["search_available_hotels", "book_hotel", "get_booking", "process_payment", "confirm_booking", "cancel_booking", "validate_booking_rules"]:
    try:
        resp = lambda_client.get_function(FunctionName=f"hotel-booking-{tool_name}")
        LAMBDA_ARNS[tool_name] = resp["Configuration"]["FunctionArn"]
    except: pass
if LAMBDA_ARNS:
    print(f"Lambda funcs:   {len(LAMBDA_ARNS)} found")
else:
    print("Lambda funcs:   not created yet")

# Gateway
GATEWAY_ID = None
try:
    gateways = agentcore.list_gateways()["items"]
    gw = next((g for g in gateways if g["name"] == GATEWAY_NAME), None)
    if gw:
        GATEWAY_ID = gw["gatewayId"]
        print(f"Gateway:        {GATEWAY_ID} ({gw['status']})")
except: 
    print("Gateway:        not created yet")

# Runtime
RUNTIME_ID = None
RUNTIME_ARN = None
try:
    runtimes = agentcore.list_agent_runtimes()["agentRuntimes"]
    rt = next((r for r in runtimes if r["agentRuntimeName"] == RUNTIME_NAME), None)
    if rt:
        RUNTIME_ID = rt["agentRuntimeId"]
        RUNTIME_ARN = rt["agentRuntimeArn"]
        print(f"Runtime:        {RUNTIME_ID} ({rt['status']})")
except:
    print("Runtime:        not created yet")

print("\nRecovery complete. You can now run any step.")

---
## Step 1: Create DynamoDB Tables

The booking system uses three tables:

| Table | Key | What it stores |
|-------|-----|----------------|
| **workshop-Hotels** | `hotel_id` | Hotel catalog: name, city, stars, price, availability |
| **workshop-Bookings** | `booking_id` | Reservations: guest, dates, status (PENDING → PAID → CONFIRMED) |
| **workshop-SteeringRules** | `rule_id` | Business rules the agent must follow (max guests, advance booking, etc.) |

All tables use **PAY_PER_REQUEST** billing — you only pay for what you use during the workshop.

In [ ]:
tables = [
    {"name": HOTELS_TABLE, "key": "hotel_id"},
    {"name": BOOKINGS_TABLE, "key": "booking_id"},
    {"name": STEERING_RULES_TABLE, "key": "rule_id"},
]

for t in tables:
    try:
        dynamodb.create_table(
            TableName=t["name"],
            KeySchema=[{"AttributeName": t["key"], "KeyType": "HASH"}],
            AttributeDefinitions=[{"AttributeName": t["key"], "AttributeType": "S"}],
            BillingMode="PAY_PER_REQUEST",
        )
        print(f"Creating {t['name']}...")
    except dynamodb.exceptions.ResourceInUseException:
        print(f"{t['name']} already exists")

# Wait for all tables to be active
for t in tables:
    waiter = dynamodb.get_waiter("table_exists")
    waiter.wait(TableName=t["name"])
    print(f"{t['name']} — ACTIVE")

print("\nAll tables ready.")

---
## Step 2: Seed Hotel Data

We populate the Hotels table with **18 hotels** across 18 cities worldwide, with prices derived from the hotel-bookings.csv dataset average daily rates by country.

Notice that **AnyCompany Rome Centro** has `available_rooms: 0` — this tests the "sold out" scenario where the agent must handle unavailability gracefully.

In [ ]:
HOTELS = [
    {"hotel_id": "anycompany-lisbon-resort", "name": "AnyCompany Lisbon Resort", "city": "Lisbon", "country": "Portugal", "stars": 4, "price_per_night": 95, "max_guests_per_room": 4, "total_rooms": 80, "available_rooms": 20},
    {"hotel_id": "anycompany-london-city", "name": "AnyCompany London City Hotel", "city": "London", "country": "UK", "stars": 4, "price_per_night": 97, "max_guests_per_room": 3, "total_rooms": 120, "available_rooms": 15},
    {"hotel_id": "anycompany-paris-central", "name": "AnyCompany Paris Central", "city": "Paris", "country": "France", "stars": 5, "price_per_night": 110, "max_guests_per_room": 4, "total_rooms": 100, "available_rooms": 12},
    {"hotel_id": "anycompany-barcelona-beach", "name": "AnyCompany Barcelona Beach", "city": "Barcelona", "country": "Spain", "stars": 4, "price_per_night": 118, "max_guests_per_room": 3, "total_rooms": 60, "available_rooms": 25},
    {"hotel_id": "anycompany-berlin-mitte", "name": "AnyCompany Berlin Mitte", "city": "Berlin", "country": "Germany", "stars": 3, "price_per_night": 105, "max_guests_per_room": 3, "total_rooms": 70, "available_rooms": 18},
    {"hotel_id": "anycompany-rome-centro", "name": "AnyCompany Rome Centro", "city": "Rome", "country": "Italy", "stars": 3, "price_per_night": 115, "max_guests_per_room": 2, "total_rooms": 45, "available_rooms": 0},
    {"hotel_id": "anycompany-dublin-temple", "name": "AnyCompany Dublin Temple Bar", "city": "Dublin", "country": "Ireland", "stars": 4, "price_per_night": 100, "max_guests_per_room": 3, "total_rooms": 50, "available_rooms": 10},
    {"hotel_id": "anycompany-brussels-grand", "name": "AnyCompany Brussels Grand Place", "city": "Brussels", "country": "Belgium", "stars": 3, "price_per_night": 114, "max_guests_per_room": 2, "total_rooms": 40, "available_rooms": 8},
    {"hotel_id": "anycompany-sao-paulo-jardins", "name": "AnyCompany São Paulo Jardins", "city": "São Paulo", "country": "Brazil", "stars": 4, "price_per_night": 112, "max_guests_per_room": 3, "total_rooms": 90, "available_rooms": 22},
    {"hotel_id": "anycompany-amsterdam-canal", "name": "AnyCompany Amsterdam Canal", "city": "Amsterdam", "country": "Netherlands", "stars": 4, "price_per_night": 108, "max_guests_per_room": 2, "total_rooms": 55, "available_rooms": 14},
    {"hotel_id": "anycompany-new-york-midtown", "name": "AnyCompany New York Midtown", "city": "New York", "country": "USA", "stars": 5, "price_per_night": 124, "max_guests_per_room": 4, "total_rooms": 150, "available_rooms": 5},
    {"hotel_id": "anycompany-zurich-lake", "name": "AnyCompany Zurich Lake", "city": "Zurich", "country": "Switzerland", "stars": 5, "price_per_night": 122, "max_guests_per_room": 3, "total_rooms": 60, "available_rooms": 9},
    {"hotel_id": "anycompany-beijing-imperial", "name": "AnyCompany Beijing Imperial", "city": "Beijing", "country": "China", "stars": 4, "price_per_night": 109, "max_guests_per_room": 3, "total_rooms": 110, "available_rooms": 30},
    {"hotel_id": "anycompany-vienna-ring", "name": "AnyCompany Vienna Ring", "city": "Vienna", "country": "Austria", "stars": 4, "price_per_night": 107, "max_guests_per_room": 3, "total_rooms": 65, "available_rooms": 16},
    {"hotel_id": "anycompany-stockholm-old", "name": "AnyCompany Stockholm Old Town", "city": "Stockholm", "country": "Sweden", "stars": 3, "price_per_night": 114, "max_guests_per_room": 2, "total_rooms": 35, "available_rooms": 7},
    {"hotel_id": "anycompany-shanghai-bund", "name": "AnyCompany Shanghai Bund", "city": "Shanghai", "country": "China", "stars": 5, "price_per_night": 111, "max_guests_per_room": 4, "total_rooms": 130, "available_rooms": 28},
    {"hotel_id": "anycompany-warsaw-royal", "name": "AnyCompany Warsaw Royal", "city": "Warsaw", "country": "Poland", "stars": 3, "price_per_night": 108, "max_guests_per_room": 2, "total_rooms": 50, "available_rooms": 12},
    {"hotel_id": "anycompany-porto-riverside", "name": "AnyCompany Porto Riverside", "city": "Porto", "country": "Portugal", "stars": 3, "price_per_night": 75, "max_guests_per_room": 2, "total_rooms": 30, "available_rooms": 15},
]

hotels_table = dynamodb_resource.Table(HOTELS_TABLE)
for hotel in HOTELS:
    hotels_table.put_item(Item=hotel)
    print(f"  {hotel['name']:30s} | {hotel['city']:15s} | {'*' * hotel['stars']:5s} | ${hotel['price_per_night']}/night | {hotel['available_rooms']} rooms")

print(f"\n{len(HOTELS)} hotels seeded.")

## Step 3: Seed Steering Rules

Steering rules are the **production version of the neurosymbolic guardrails** from Module 3. Instead of being hardcoded in Python, they live in DynamoDB — you can change them without redeploying the agent.

Each rule has:
- **`fail_message`**: What the agent reports when the rule is violated
- **`steer_message`**: How the agent should self-correct (the STEER instruction from Module 4)

| Rule | Action | Condition | What Happens |
|------|--------|-----------|-------------|
| max-guests | book | guests > 10 | STEER: adjust to 10 guests |
| valid-dates | book | nights < 1 | STEER: swap check-in/check-out |
| advance-booking | book | same-day | STEER: move to tomorrow |
| payment-before-confirm | confirm | status != PAID | STEER: process payment first |
| cancellation-window | cancel | < 48h before check-in | STEER: offer modification |
| already-cancelled | cancel | status == CANCELLED | STEER: offer new reservation |

In [ ]:
STEERING_RULES = [
    {"rule_id": "max-guests", "action": "book", "condition_field": "guests", "operator": "gt", "threshold": 10,
     "fail_message": "Guest count exceeds maximum of 10",
     "steer_message": "The booking exceeds the hotel maximum of 10 guests per room. Calculate how many rooms needed (divide {guests} by 10, rounding up). Do NOT describe or explain — immediately call book_hotel multiple times: fill rooms with 10 guests each until all guests are accommodated. After all calls succeed, tell the user their reservation was split into the appropriate number of rooms at the same hotel and dates.",
     "enabled": True},
    {"rule_id": "valid-dates", "action": "book", "condition_field": "nights", "operator": "lt", "threshold": 1,
     "fail_message": "Check-in date must be before check-out date",
     "steer_message": "The dates appear reversed. Swap check-in and check-out, proceed with the booking, and tell the user.",
     "enabled": True},
    {"rule_id": "advance-booking", "action": "book", "condition_field": "days_until_checkin", "operator": "lt", "threshold": 1,
     "fail_message": "Booking must be made at least 1 day in advance",
     "steer_message": "Same-day bookings are not available. Adjust the check-in to tomorrow's date and proceed.",
     "enabled": True},
    {"rule_id": "payment-before-confirm", "action": "confirm", "condition_field": "booking_status", "operator": "ne", "threshold": "PAID",
     "fail_message": "Payment must be processed before confirmation",
     "steer_message": "Confirmation requires payment first. Process the payment, then confirm.",
     "enabled": True},
    {"rule_id": "cancellation-window", "action": "cancel", "condition_field": "days_until_checkin", "operator": "lt", "threshold": 2,
     "fail_message": "Cannot cancel within 48 hours of check-in",
     "steer_message": "Cancellation within 48 hours is not available. Offer to modify the reservation instead.",
     "enabled": True},
    {"rule_id": "already-cancelled", "action": "cancel", "condition_field": "booking_status", "operator": "eq", "threshold": "CANCELLED",
     "fail_message": "Booking is already cancelled",
     "steer_message": "This booking is already cancelled. Offer to make a new reservation.",
     "enabled": True},
]

rules_table = dynamodb_resource.Table(STEERING_RULES_TABLE)
for rule in STEERING_RULES:
    rules_table.put_item(Item=rule)
    print(f"  [{rule['action']:7s}] {rule['rule_id']:25s} — {rule['fail_message']}")

print(f"\n{len(STEERING_RULES)} steering rules seeded.")

---
## Step 4: Create IAM Roles

We need two IAM roles:

1. **Lambda Execution Role** — allows the 7 Lambda functions to read/write DynamoDB and write CloudWatch logs
2. **AgentCore Execution Role** — allows the agent runtime to invoke Bedrock models, call Lambda tools via the Gateway, and read bookings for guardrail validation

Both roles use **least-privilege** policies scoped to the specific resources created in this workshop.

In [ ]:
# --- Lambda Execution Role ---
# This role is used by ALL Lambda functions (booking tools + Neo4j query).
# It includes DynamoDB access, CloudWatch logs, VPC networking (for Neo4j Lambda),
# and Secrets Manager access (to retrieve the Neo4j password).

lambda_trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
})

lambda_statements = [
    {
        "Effect": "Allow",
        "Action": ["dynamodb:GetItem", "dynamodb:PutItem", "dynamodb:UpdateItem", "dynamodb:Scan", "dynamodb:Query"],
        "Resource": [
            f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{HOTELS_TABLE}",
            f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{BOOKINGS_TABLE}",
            f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{STEERING_RULES_TABLE}",
        ]
    },
    {
        "Effect": "Allow",
        "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
        "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/lambda/hotel-booking-*"
    },
]

# Add VPC networking permissions (needed for Neo4j Lambda to connect to EC2)
if NEO4J_HOST and SECURITY_GROUP_ID:
    lambda_statements.append({
        "Effect": "Allow",
        "Action": [
            "ec2:CreateNetworkInterface",
            "ec2:DescribeNetworkInterfaces",
            "ec2:DeleteNetworkInterface",
        ],
        "Resource": "*"
    })

# Add Secrets Manager access (Neo4j password)
if NEO4J_SECRET_ARN:
    lambda_statements.append({
        "Effect": "Allow",
        "Action": ["secretsmanager:GetSecretValue"],
        "Resource": NEO4J_SECRET_ARN
    })

lambda_policy = json.dumps({"Version": "2012-10-17", "Statement": lambda_statements})

try:
    iam.create_role(RoleName=LAMBDA_ROLE_NAME, AssumeRolePolicyDocument=lambda_trust, Description="Lambda role for hotel booking tools")
    print(f"Created {LAMBDA_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"{LAMBDA_ROLE_NAME} already exists — updating policy")

iam.put_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyName="DynamoDBAndLogs", PolicyDocument=lambda_policy)
iam.attach_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")

LAMBDA_ROLE_ARN = iam.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]
print(f"ARN: {LAMBDA_ROLE_ARN}")
if NEO4J_HOST:
    print(f"  + VPC networking permissions (Neo4j Lambda)")
    print(f"  + Secrets Manager access ({NEO4J_SECRET_ARN})")

In [ ]:
# --- AgentCore Execution Role ---

agentcore_trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock-agentcore.amazonaws.com"}, "Action": "sts:AssumeRole"}]
})

agentcore_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["lambda:InvokeFunction"], "Resource": f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:hotel-booking-*"},
        {"Effect": "Allow", "Action": ["dynamodb:GetItem"], "Resource": f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{BOOKINGS_TABLE}"},
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"], "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/*"},
        {"Effect": "Allow", "Action": ["bedrock-agentcore:GetGateway", "bedrock-agentcore:GetGatewayTarget", "bedrock-agentcore:ListGatewayTargets", "bedrock-agentcore:InvokeGateway"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["ecr:GetAuthorizationToken"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer", "ecr:BatchCheckLayerAvailability"], "Resource": f"arn:aws:ecr:{REGION}:{ACCOUNT_ID}:repository/bedrock-agentcore-*"},
    ]
})

try:
    iam.create_role(RoleName=AGENTCORE_ROLE_NAME, AssumeRolePolicyDocument=agentcore_trust, Description="AgentCore role for hotel booking agent")
    print(f"Created {AGENTCORE_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"{AGENTCORE_ROLE_NAME} already exists — updating policy")

# Always update the policy to ensure ECR permissions are included
iam.put_role_policy(RoleName=AGENTCORE_ROLE_NAME, PolicyName="AgentCorePermissions", PolicyDocument=agentcore_policy)
iam.attach_role_policy(RoleName=AGENTCORE_ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/AWSXRayDaemonWriteAccess")

AGENTCORE_ROLE_ARN = iam.get_role(RoleName=AGENTCORE_ROLE_NAME)["Role"]["Arn"]
print(f"ARN: {AGENTCORE_ROLE_ARN}")

print("\nWaiting 10s for IAM propagation...")
time.sleep(10)
print("Ready.")

---
## Step 5: Deploy Lambda Functions

Each Lambda function is a **booking tool** that the agent can call through the Gateway:

| Function | What it does | Key DynamoDB operations |
|----------|-------------|------------------------|
| `search_available_hotels` | Find hotels by city, country, price, stars | Scan with filters |
| `book_hotel` | Create a reservation (status: PENDING) | PutItem + UpdateItem (decrement rooms) |
| `get_booking` | Retrieve booking details | GetItem |
| `process_payment` | Pay for a booking (PENDING → PAID) | UpdateItem |
| `confirm_booking` | Confirm after payment (PAID → CONFIRMED) | UpdateItem |
| `cancel_booking` | Cancel and return room to inventory | UpdateItem (both tables) |
| `validate_booking_rules` | Check steering rules from DynamoDB | Scan rules + evaluate |

The code for each function lives in `lambda_tools/<name>/lambda_function.py`. We zip each one and deploy it.

In [ ]:
LAMBDA_TOOLS = [
    "search_available_hotels", "book_hotel", "get_booking",
    "process_payment", "confirm_booking", "cancel_booking",
    "validate_booking_rules",
]

if 'LAMBDA_ARNS' not in dir() or not LAMBDA_ARNS:
    LAMBDA_ARNS = {}

for tool_name in LAMBDA_TOOLS:
    function_name = f"hotel-booking-{tool_name}"

    # Zip the Lambda code
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(f"lambda_tools/{tool_name}/lambda_function.py", "lambda_function.py")
    zip_bytes = zip_buffer.getvalue()

    try:
        resp = lambda_client.create_function(
            FunctionName=function_name,
            Runtime="python3.11",
            Role=LAMBDA_ROLE_ARN,
            Handler="lambda_function.handler",
            Code={"ZipFile": zip_bytes},
            Timeout=30,
            MemorySize=256,
            Environment={"Variables": {
                "HOTELS_TABLE": HOTELS_TABLE,
                "BOOKINGS_TABLE": BOOKINGS_TABLE,
                "STEERING_RULES_TABLE": STEERING_RULES_TABLE,
            }},
        )
        LAMBDA_ARNS[tool_name] = resp["FunctionArn"]
        print(f"  Created  {function_name}")
    except lambda_client.exceptions.ResourceConflictException:
        lambda_client.update_function_code(FunctionName=function_name, ZipFile=zip_bytes)
        resp = lambda_client.get_function(FunctionName=function_name)
        LAMBDA_ARNS[tool_name] = resp["Configuration"]["FunctionArn"]
        print(f"  Updated  {function_name}")

# Wait for all to be active
for tool_name in LAMBDA_TOOLS:
    waiter = lambda_client.get_waiter("function_active_v2")
    waiter.wait(FunctionName=f"hotel-booking-{tool_name}")

print(f"\n{len(LAMBDA_ARNS)} Lambda functions deployed and active.")

### Deploy Neo4j Query Lambda (VPC)

The `query_knowledge_graph` Lambda connects to the Neo4j instance running on the Code Editor EC2.
Unlike the booking Lambdas, this one runs **inside the VPC** so it can reach the EC2 private IP on port 7687.

It also needs the `neo4j` Python driver as a dependency, which we package as a Lambda Layer.

In [ ]:
import sys
# Deploy Neo4j query Lambda — only if Neo4j infrastructure was detected
if NEO4J_HOST and SECURITY_GROUP_ID and SUBNET_IDS:
    print("Deploying Neo4j query Lambda with VPC config...")

    # Step 1: Create a Lambda Layer with the neo4j Python driver
    # The neo4j package is not in the Lambda runtime — we need to bundle it
    import shutil, subprocess

    layer_dir = "/tmp/neo4j-layer/python"
    if os.path.exists("/tmp/neo4j-layer"):
        shutil.rmtree("/tmp/neo4j-layer", ignore_errors=True)
    os.makedirs(layer_dir)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "neo4j", "-t", layer_dir, "--quiet"],
        check=True, capture_output=True,
    )

    layer_zip = io.BytesIO()
    with zipfile.ZipFile(layer_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk("/tmp/neo4j-layer"):
            for f in files:
                fp = os.path.join(root, f)
                arcname = os.path.relpath(fp, "/tmp/neo4j-layer")
                zf.write(fp, arcname)
    layer_bytes = layer_zip.getvalue()
    print(f"  Neo4j layer: {len(layer_bytes) / 1024 / 1024:.1f} MB")

    # Create or update the Lambda Layer
    try:
        layer_resp = lambda_client.publish_layer_version(
            LayerName="workshop-neo4j-driver",
            Description="Neo4j Python driver for query_knowledge_graph Lambda",
            Content={"ZipFile": layer_bytes},
            CompatibleRuntimes=["python3.11", "python3.12", "python3.13"],
        )
        LAYER_ARN = layer_resp["LayerVersionArn"]
        print(f"  Layer: {LAYER_ARN}")
    except Exception as e:
        print(f"  Layer error: {e}")
        LAYER_ARN = None

    # Step 2: Deploy the Lambda function with VPC config
    function_name = "hotel-booking-query_knowledge_graph"
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("lambda_tools/query_knowledge_graph/lambda_function.py", "lambda_function.py")
    zip_bytes = zip_buffer.getvalue()

    lambda_config = {
        "FunctionName": function_name,
        "Runtime": "python3.11",
        "Role": LAMBDA_ROLE_ARN,
        "Handler": "lambda_function.handler",
        "Code": {"ZipFile": zip_bytes},
        "Timeout": 30,
        "MemorySize": 256,
        "Environment": {"Variables": {
            "NEO4J_HOST": NEO4J_HOST,
            "NEO4J_PASSWORD_SECRET_ARN": NEO4J_SECRET_ARN,
        }},
        "VpcConfig": {
            "SubnetIds": SUBNET_IDS,
            "SecurityGroupIds": [SECURITY_GROUP_ID],
        },
    }
    if LAYER_ARN:
        lambda_config["Layers"] = [LAYER_ARN]

    try:
        resp = lambda_client.create_function(**lambda_config)
        LAMBDA_ARNS["query_knowledge_graph"] = resp["FunctionArn"]
        print(f"  Created {function_name} (VPC: {SECURITY_GROUP_ID})")
    except lambda_client.exceptions.ResourceConflictException:
        lambda_client.update_function_code(FunctionName=function_name, ZipFile=zip_bytes)
        resp = lambda_client.get_function(FunctionName=function_name)
        LAMBDA_ARNS["query_knowledge_graph"] = resp["Configuration"]["FunctionArn"]
        print(f"  Updated {function_name}")

    # Wait for it to be active
    waiter = lambda_client.get_waiter("function_active_v2")
    waiter.wait(FunctionName=function_name)
    print(f"  {function_name} — ACTIVE")

else:
    print("Neo4j not detected — skipping query_knowledge_graph Lambda.")
    print("The agent will work with booking tools only (no graph queries).")
    print("To enable: set NEO4J_HOST, NEO4J_SECRET_ARN, SECURITY_GROUP_ID, SUBNET_IDS")

print(f"\nTotal Lambda functions: {len(LAMBDA_ARNS)}")

---
## Step 6: Create AgentCore Gateway

The **AgentCore Gateway** is the production version of the FAISS-based semantic tool filtering from Module 1.

Instead of building a local FAISS index, the Gateway uses **MCP (Model Context Protocol)** with built-in **semantic search** to route requests to the right Lambda tool based on the user's query.

For example:
- "Find me a hotel in Paris" → routes to `search_available_hotels`
- "Pay for my booking" → routes to `process_payment`
- "Is my booking valid?" → routes to `validate_booking_rules`

In [ ]:
try:
    resp = agentcore.create_gateway(
        name=GATEWAY_NAME,
        description="Hotel booking tools: search, book, pay, confirm, cancel, and validate business rules.",
        protocolType="MCP",
        protocolConfiguration={
            "mcp": {
                "instructions": "Hotel booking tools: search, book, pay, confirm, cancel, and validate business rules.",
                "searchType": "SEMANTIC",
                "supportedVersions": ["2025-03-26"],
            }
        },
        authorizerType="NONE",
        roleArn=AGENTCORE_ROLE_ARN,
    )
    GATEWAY_ID = resp["gatewayId"]
    print(f"Created gateway: {GATEWAY_ID}")
except Exception as e:
    if "ConflictException" in str(type(e)) or "already exists" in str(e).lower():
        gateways = agentcore.list_gateways()["items"]
        GATEWAY_ID = next(g["gatewayId"] for g in gateways if g["name"] == GATEWAY_NAME)
        print(f"Gateway already exists: {GATEWAY_ID}")
    else:
        raise

# Wait for gateway to be ready
print("Waiting for gateway...")
for _ in range(30):
    gw = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
    status = gw.get("status", "UNKNOWN")
    if status in ("READY", "ACTIVE", "CREATE_COMPLETE"):
        break
    time.sleep(5)

print(f"Gateway status: {status}")
print(f"Gateway ID: {GATEWAY_ID}")

## Step 7: Register Lambda Tools as Gateway Targets

Each Lambda function is registered as a **Gateway Target** with its tool schema. The Gateway uses these schemas for **semantic routing** — matching user queries to the right tool automatically.

All tool schemas are loaded from `tool_schemas/tools.json`. If Neo4j was detected, the `query_knowledge_graph` tool is included; otherwise it is skipped.

In [ ]:
# Load all tool schemas
with open("tool_schemas/tools.json") as f:
    all_schemas = json.load(f)

# Include query_knowledge_graph only if we deployed its Lambda
available_tools = set(LAMBDA_ARNS.keys())
tool_schemas = {s["name"]: s for s in all_schemas if s["name"] in available_tools}

print(f"Registering {len(tool_schemas)} tools as Gateway targets:")
for name in tool_schemas:
    print(f"  - {name}" + (" (Neo4j VPC)" if name == "query_knowledge_graph" else ""))

# Clean schema properties to only include fields the API accepts
ALLOWED_PROP_FIELDS = {"type", "description", "properties", "required", "items"}

def clean_schema(schema_obj):
    """Recursively clean schema to only include API-accepted fields."""
    if not isinstance(schema_obj, dict):
        return schema_obj
    cleaned = {}
    for k, v in schema_obj.items():
        if k == "properties" and isinstance(v, dict):
            cleaned[k] = {pk: clean_schema(pv) for pk, pv in v.items()}
        elif k in ALLOWED_PROP_FIELDS:
            cleaned[k] = clean_schema(v) if isinstance(v, dict) else v
    return cleaned

print()
for tool_name, lambda_arn in LAMBDA_ARNS.items():
    target_name = tool_name.replace("_", "-")
    schema = tool_schemas.get(tool_name)
    if not schema:
        continue

    cleaned_input = clean_schema(schema["input_schema"])

    try:
        agentcore.create_gateway_target(
            name=target_name,
            gatewayIdentifier=GATEWAY_ID,
            credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
            targetConfiguration={
                "mcp": {
                    "lambda": {
                        "lambdaArn": lambda_arn,
                        "toolSchema": {
                            "inlinePayload": [{
                                "name": schema["name"],
                                "description": schema["description"],
                                "inputSchema": cleaned_input,
                            }]
                        },
                    }
                }
            },
        )
        print(f"  Registered: {target_name}")
    except Exception as e:
        if "ConflictException" in str(type(e)) or "already exists" in str(e).lower():
            print(f"  Already exists: {target_name}")
        else:
            print(f"  ERROR: {target_name}: {e}")
            raise

print(f"\n{len(LAMBDA_ARNS)} gateway targets configured.")

---
## Step 8: Deploy AgentCore Runtime

Now we deploy the **Strands agent** to AgentCore using the `bedrock-agentcore-starter-toolkit`.

The toolkit handles:
- Building the deployment package for ARM64 architecture
- Creating the ECR repository and Docker image
- Creating the AgentCore Runtime with proper configuration

The agent code (`booking_agent.py`) connects to the Gateway via MCP, uses Bedrock Claude as the model, and includes hard guardrails via lifecycle hooks.

> **This step takes 3-5 minutes.** The toolkit builds a container image and deploys it to AgentCore.

In [ ]:
# Pre-flight cleanup — delete any leftover CodeBuild project and config from previous runs
# This prevents ResourceAlreadyExistsException when running the cell below
import os, glob

CB_PROJECT = f"bedrock-agentcore-{RUNTIME_NAME.lower()}-builder"

# Delete leftover CodeBuild project
try:
    codebuild_client = boto3.client("codebuild", region_name=REGION)
    codebuild_client.delete_project(name=CB_PROJECT)
    print(f"Deleted previous CodeBuild project: {CB_PROJECT}")
except Exception:
    pass  # Does not exist — that's fine

# Delete starter toolkit config with old runtime ID
for cfg in glob.glob(".bedrock_agentcore*.yaml") + glob.glob(os.path.expanduser("~/.bedrock_agentcore*.yaml")):
    try:
        os.remove(cfg)
        print(f"Deleted stale config: {cfg}")
    except Exception:
        pass

print("✅ Pre-flight cleanup done")

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Recover role ARN
AGENTCORE_ROLE_ARN = iam.get_role(RoleName=AGENTCORE_ROLE_NAME)["Role"]["Arn"]
print(f"Role: {AGENTCORE_ROLE_ARN}")

# Get Gateway URL
if not GATEWAY_ID:
    raise ValueError("GATEWAY_ID not set. Run the recovery cell or the gateway creation cell first.")
gw_info = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
GATEWAY_URL = gw_info["gatewayUrl"]
print(f"Gateway URL: {GATEWAY_URL}")

# Configure the agent
# NOTE: Do NOT set protocol="MCP" here. The Gateway handles MCP.
# The runtime is a standard code deployment that receives payloads via invoke_agent_runtime.
agent_runtime = Runtime()

agent_runtime.configure(
    entrypoint="booking_agent.py",
    execution_role=AGENTCORE_ROLE_ARN,
    auto_create_ecr=True,
    requirements_file="agent_requirements.txt",
    region=REGION,
    agent_name=RUNTIME_NAME,
    memory_mode="NO_MEMORY",
    deployment_type="container",
    non_interactive=True,
)

print("\nLaunching agent (3-5 minutes)...")

result = agent_runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "AWS_REGION": REGION,
        "BOOKINGS_TABLE": BOOKINGS_TABLE,
        "GATEWAY_URL": GATEWAY_URL,
    },
)

RUNTIME_ARN = result.agent_arn
RUNTIME_ID = RUNTIME_ARN.split("/")[-1] if RUNTIME_ARN else None
print(f"\nAgent deployed: {RUNTIME_ARN}")

---
## Step 9: Test the Agent

Each test exercises different capabilities of the production agent:

| Test | What it exercises |
|------|-------------------|
| 1. Search hotels in Lisbon | Semantic routing → `search_available_hotels` Lambda → DynamoDB scan |
| 2. Book for 15 guests | Steering rule: max 10 guests → agent self-corrects to 10 |
| 3. Full booking flow | Multi-turn session: search → book → pay → confirm (same session ID) |
| 4. Sold out hotel | AnyCompany Rome Centro has 0 rooms → agent handles gracefully |
| 5. Budget search | Cross-country search under $100 → finds cheapest hotels |
| 6. Confirm without payment | Hard guardrail (BookingGuardrailsHook) → BLOCKED |
| 7. Same-day booking | Steering rule: advance booking required → agent adjusts to tomorrow |

In [ ]:
# Test 5b: Neo4j Knowledge Graph Query
# Forces the agent to use query_knowledge_graph Lambda (Graph-RAG from Module 1)
# This query requires aggregation across multiple hotel entities — only Neo4j can answer it
invoke_agent("What is the average guest rating across all AnyCompany hotels in Europe?")

In [ ]:
import uuid

def invoke_agent(prompt, session_id=None):
    """Invoke the deployed agent and print the response."""
    if not session_id:
        session_id = str(uuid.uuid4())

    print(f"\nUser: {prompt}")
    print("-" * 60)

    try:
        # Payload format matching the agent's expected input: payload.get("prompt", "")
        payload = json.dumps({"prompt": prompt}).encode()

        response = agentcore_data.invoke_agent_runtime(
            agentRuntimeArn=RUNTIME_ARN,
            runtimeSessionId=session_id,
            payload=payload,
        )

        # Read streaming response
        content = []
        for chunk in response.get("response", []):
            content.append(chunk.decode("utf-8"))

        result = "".join(content)

        # Try to parse as JSON if possible
        try:
            result_json = json.loads(result)
            if isinstance(result_json, dict) and "response" in result_json:
                result = result_json["response"]
        except (json.JSONDecodeError, TypeError):
            pass

        print(f"Agent: {result}")
        return result, session_id

    except Exception as e:
        print(f"Error: {e}")
        return None, session_id

In [ ]:
# Test 1: Search for hotels — exercises semantic routing + DynamoDB scan
invoke_agent("Search for hotels in Lisbon")

In [ ]:
# Test 2: Steering rule — max 10 guests
# The agent should self-correct by splitting into 2 rooms (10 + 5) instead of blocking
invoke_agent("Book AnyCompany Paris Central for 15 guests from 2026-06-01 to 2026-06-05")

In [ ]:
# Test 3: Full booking flow in a single session
# search → validate → book → pay → confirm
session_id = str(uuid.uuid4())

print("=== Full Booking Flow (single session) ===\n")

# Step 1: Search
invoke_agent("Find me a 4-star hotel in London", session_id=session_id)

In [ ]:
# Step 2: Book (same session — agent remembers the hotel from the search)
invoke_agent("Book AnyCompany London City Hotel for 2 guests from 2026-07-10 to 2026-07-14", session_id=session_id)

In [ ]:
# Step 3: Pay and confirm (same session — agent knows the booking ID)
invoke_agent("Pay for my booking and confirm it", session_id=session_id)

In [ ]:
# Test 4: Sold out hotel — AnyCompany Rome Centro has 0 rooms
invoke_agent("Book AnyCompany Rome Centro for 2 guests from 2026-08-01 to 2026-08-03")

In [ ]:
# Test 5: Budget search across countries
invoke_agent("Find hotels under $100 per night")

In [ ]:
# Test 6: Hard guardrail — confirm without payment
# The BookingGuardrailsHook BLOCKS this at the framework level
invoke_agent("Confirm booking BK-DOESNOTEXIST")

In [ ]:
# Test 7: Steering rule — same-day booking
# The agent should adjust the check-in to tomorrow
from datetime import datetime
today = datetime.now().strftime("%Y-%m-%d")
invoke_agent(f"Book AnyCompany Amsterdam Canal for 2 guests from {today} to {today}")

---
## Summary

You deployed a production agent with:

| Component | Resource | Purpose |
|-----------|----------|----------|
| Data | 3 DynamoDB tables | Hotels, bookings, steering rules |
| Tools | 7 Lambda functions | Booking operations |
| Routing | AgentCore Gateway (MCP) | Semantic tool discovery |
| Agent | AgentCore Runtime | Strands agent with Bedrock |
| Guardrails | Hooks + DynamoDB rules | Hard blocks + soft steering |

The techniques from previous modules now work together in production:
- **Module 1** (semantic tool selection) → Gateway with MCP semantic search
- **Module 3** (neurosymbolic guardrails) → `validate_booking_rules` Lambda + DynamoDB rules
- **Module 4** (steering) → STEER messages in DynamoDB, agent self-corrects

### What's Next

- **Change a steering rule** in DynamoDB (e.g., set max guests to 5) and test again — no redeployment needed
- **Add a new tool** by creating a Lambda and registering it as a Gateway target
- **Monitor** the agent via CloudWatch logs

---
## Cleanup

Run the cell below to delete all resources created in this notebook.

> **Skip this if you are using an AWS-provided workshop account** — it will be cleaned up automatically.

In [ ]:
# Uncomment and run to delete all resources:
# !python3 cleanup.py